In [ ]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
import re
import torch
from sklearn.metrics.pairwise import cosine_similarity
import textwrap

### Replace path here 

In [ ]:
path = '/Users/madhu/Desktop/WFP/wfp-taxonomy-analysis/'

In [ ]:
abstract = pd.read_csv(path+"data/partial_results.csv")
metadata = pd.read_csv(path+"/data/metadata.csv")

In [ ]:
abstract_topic = pd.merge(metadata[['report_id','topic']], abstract.rename(columns={'id':'report_id'}), how = 'left')

### Step 1 : Convert Topics and Definitions to Embeddings

In [ ]:
taxonomy = {
    "Academia_and_think_tanks": "Engagements with universities, research institutions, and policy think tanks to generate evidence, conduct joint studies, shape food security policy, and develop analytical frameworks. It includes collaborative research, expert consultations, knowledge exchange, and externally validated methodologies, excluding internal operational analysis not involving academic or think tank partnerships.",
    "Achievements_and_history": "Institutional milestones, documented impacts, strategic accomplishments, and historical evolution of programmes, operations, or organizational initiatives. It captures past results and legacy contributions, excluding forward-looking strategy or projections.",
    "Ambassadors_and_Celebrities": "Partnerships, advocacy campaigns, and public engagement initiatives involving high-profile public figures serving as goodwill representatives or influencers to advance humanitarian messaging, awareness, and fundraising. It excludes routine political or diplomatic representation.",
    "Analyses_and_assessments": "Evidence-based studies, situation analyses, programme assessments, and structured evaluations of humanitarian or food security interventions using quantitative or qualitative methodologies. It excludes monitoring dashboards unless part of a formal assessment report.",
    "Anticipatory_Action": "Pre-crisis interventions triggered by risk forecasts to mitigate humanitarian impact before shocks occur. It includes Anticipatory Action (AA) frameworks, early financing, and pre-positioned support, excluding standard emergency response activated only after a crisis.",
    "Cash_transfers": "Direct monetary assistance to beneficiaries to improve food security, resilience, or emergency recovery. It covers cash-based transfer systems such as Cash-Based Transfers (CBT) and Conditional Cash Transfers (CCT), excluding in-kind food distribution or asset-linked conditional programmes.",
    "Centre_of_Excellence": "A Centre of Excellence (COE) is a specialized hub that develops standards, builds capacity, drives innovation, and provides technical leadership for humanitarian, logistics, analytics, or nutrition domains. COE initiatives exclude general programme execution unless anchored in COE-led methodology, training, or systems development.",
    "Climate_action": "Interventions aimed at reducing climate impact through mitigation, adaptation, policy alignment, and climate-linked financing. It includes National Adaptation Plans (NAPs), emissions mitigation efforts, and climate advocacy, excluding agricultural monitoring unless tied to climate action programmes.",
    "Climate_change_adaptation": "Strategies and programmes that help communities, systems, or supply chains adjust to climate variability, shocks, and long-term climate effects. It includes adaptation planning, resilient infrastructure, and climate-smart systems, excluding risk financing unless specifically tied to adaptation funding.",
    "Climate_risk_management_insurance_and_financing": "Climate Risk Management (CRM) insurance and financing mechanisms that reduce vulnerability using climate-linked financial instruments, forecast financing, risk insurance, or resilience funds. It excludes general climate adaptation unless tied to financial risk instruments.",
    "Climate_services": "Climate data, forecasts, early warning systems, and analytical climate tools used to guide humanitarian or food security decision-making. It excludes seasonal agriculture monitoring unless explicitly climate-tool driven.",
    "Conflicts": "Situations of armed violence, insecurity, or geopolitical instability that disrupt food security, humanitarian access, or supply chains. It captures conflict context, excluding disaster or climate emergencies unless driven by conflict.",
    "Corporate_strategy": "Organization-level strategic planning, institutional priorities, policy direction, and enterprise frameworks guiding long-term humanitarian and operational goals. It excludes country-level strategy or programme design.",
    "Country_Capacity_Strengthening": "Country Capacity Strengthening (CCS) involves government and national system enablement, technical training, institutional strengthening, and policy support to improve sustainable humanitarian and food security operations. CCS excludes direct beneficiary assistance unless part of national capacity building systems.",
    "Country_strategic_planning": "Country-level strategic frameworks, planning documents, and national alignment roadmaps guiding food security, emergency response, or resilience priorities. It excludes corporate-level strategy.",
    "Conflicts": "Contexts involving war, armed violence, political instability, or prolonged insecurity that disrupt humanitarian access, food systems, or national stability, excluding natural disasters or economic shocks unless directly driven by conflict.",
    "Emergencies": "Sudden or ongoing crisis situations requiring urgent humanitarian intervention to prevent food insecurity, mortality, or systemic collapse. It includes natural disasters, conflict-driven crises, and economic shocks when they demand immediate response, excluding long-term resilience programming.",
    "Emergency_Preparedness_and_Response": "Emergency Preparedness and Response (EPR) refers to systems, planning, and operational mechanisms enabling rapid crisis action, including risk mapping, early warning, contingency planning, response protocols, and surge capacity, excluding anticipatory financing unless forecast-triggered.",
    "Emergency_programming": "Operational design and execution of emergency food, cash, logistics, or multisectoral programmes activated due to crises. It excludes unconditional relief not tied to structured emergency programme implementation.",
    "Emergency_relief": "Unconditional life-saving assistance delivered during crises to address immediate needs such as hunger, shelter, or survival, excluding conditional or asset-linked support.",
    "Energy_for_food_security": "Energy interventions that enable or protect food security, such as renewable micro-grids, fuel supply for food logistics, energy-enabled irrigation, and power for food systems, excluding general energy infrastructure not linked to food outcomes.",
    "Engineering_services": "Technical and operational Engineering Services (ES) supporting infrastructure, construction, environmental safety, logistics facilities, or resilient asset creation. It excludes engineering unless tied to operational humanitarian support.",
    "Environmental_and_social_sustainability": "Environmental and Social Sustainability (ESS) includes policies, safeguards, sustainability frameworks, compliance, environmental impact reduction, community equity, and responsible programming, excluding SDG mapping unless tied to sustainability compliance.",
    "Executive_Board": "Executive Board (EB) refers to governance directives, decisions, policy approvals, strategic resolutions, and oversight discussions made by the formal UN Executive Board, excluding country-specific planning unless derived from EB directives.",
    "Food_Assistance": "Food Assistance (FA) refers to programmes delivering food or cash-based food support to address hunger, improve nutrition, or strengthen resilience. It includes both in-kind and cash-based food support systems unless conditional, excluding school meals unless part of a broader FA programme.",
    "Food_Assistance_for_Assets": "Food Assistance for Assets (FFA) programmes provide conditional support in the form of food or cash to beneficiaries in exchange for participation in building community assets such as roads, water systems, or climate-resilient infrastructure, excluding unconditional relief.",
    "Food_Security_Cluster": "Food Security Cluster (FSC) refers to the inter-agency coordination mechanism co-led by WFP/FAO during emergencies to align food security response, partner coordination, gap assessment, resource alignment, and joint response planning, excluding independent evaluations not tied to FSC coordination.",
    "Food_fortification": "Food Fortification (FF) involves adding micronutrients to staple foods to improve nutritional outcomes at population scale, excluding specialized nutritious food formulations that are not mass fortification.",
    "Food_safety_and_quality": "Food Safety and Quality (FSQ) covers standards, compliance, testing, storage, and safeguards ensuring food distributed is safe, nutritious, and meets regulatory and humanitarian quality thresholds, excluding procurement unless tied to safety compliance.",
    "Food_security_analysis": "Food Security Analysis (FSA) includes evidence-driven hunger assessments, food insecurity classification, Integrated Food Security Phase Classification (IPC), vulnerability mapping, nutrition linkage analysis, and consumption pattern analysis, excluding local market analysis unless explicitly part of FSA evidence.",
    "Food_systems": "Food Systems (FS) include national and community food production, distribution, consumption, policy, sustainability, supply chain, resilience, and market interactions that influence systemic food security, excluding logistics unless tied to FS system enablement.",
    "Funding_and_donors": "Funding and Donors (F&D) includes financial partnerships, donor reporting, grants, funding pipelines, pledges, and institutional financing that enable food security or humanitarian operations, excluding internal procurement budgets not tied to donor engagement.",
    "Gender_equality": "Gender Equality (GE) involves policies and programmes ensuring equitable access, participation, outcomes, and empowerment for all genders within humanitarian and food security operations, excluding advocacy unless tied to GE programme implementation.",
    "Governance_and_leadership": "Governance and Leadership (G&L) refers to institutional decision-making, leadership structures, accountability, compliance, policy oversight, and strategic governance systems, excluding Executive Board unless directly tied to EB governance outcomes.",
    "HIV_and_tuberculosis": "HIV and Tuberculosis (HIV/TB) includes nutrition, treatment support, supply chain enablement, food assistance, healthcare access programming, and vulnerability mitigation for populations affected by HIV or TB, excluding general nutrition unless tied to HIV/TB programme support.",
    "Humanitarian_Support_and_Services": "Humanitarian Support and Services (HSS) includes logistics, telecommunications, air support, engineering, emergency coordination, surge services, and enabling systems supporting humanitarian operations, excluding direct beneficiary aid.",
    "In_kind_food_distribution": "In-Kind Food Distribution (IKD) refers to the physical delivery of food commodities, baskets, or nutrition supplies to beneficiaries, excluding cash-based or conditional transfer programmes.",
    "Independent_evaluation": "Independent Evaluation (IE) refers to externally or institutionally independent assessment of programmes, policies, or impact with no operational or programme execution ownership, excluding internal monitoring.",
    "Information_Technology_and_Telecommunications_in_Emergencies": "Information Technology and Telecommunications in Emergencies (ITTE) includes emergency ICT systems, connectivity, crisis communication infrastructure, satellite comms, and digital response systems enabling humanitarian operations, excluding routine IT upgrades.",
    "Innovation": "Innovation involves pilot testing, scaling new digital or programme solutions such as AI, drones, blockchain, or digital delivery models that improve humanitarian or food security outcomes, excluding routine IT infrastructure maintenance.",
    "Local_market_developments": "Local Market Developments (LMD) captures food availability, demand, price movements, market disruptions, and evolving local market behavior that impact food security or beneficiary purchasing power, excluding national Market Analysis unless LMD context-specific.",
    "Logistics_and_delivery_networks": "Logistics and Delivery Networks (LDN) refer to systems enabling movement, delivery, warehousing, last-mile distribution, and network efficiency of humanitarian supplies, excluding procurement unless tied to delivery network enablement.",
    "Market_analysis": "Market Analysis (MA) includes national or regional price trends, demand forecasting, supply mapping, trade flow analysis, market health, commodity risk analysis, and systemic market insights affecting food or humanitarian operations, excluding local price monitoring unless part of MA evidence.",
    "Migration": "Migration includes displacement, refugee movement, cross-border mobility, internal relocation, and food or vulnerability outcomes tied to migration or displaced populations, excluding conflict unless migration directly conflict-driven.",
    "Monitoring": "Monitoring (MON) includes indicator tracking, data collection, progress measurement, field monitoring systems, dashboards, and operational performance tracking, excluding formal programme evaluation unless tied to monitoring frameworks.",
    "Monitoring_evaluation_and_learning": "Monitoring, Evaluation and Learning (MEL) includes frameworks, indicators, evaluation insights, and learning loops enabling programme improvement, excluding standalone Monitoring unless tied to MEL learning outcomes.",
    "Nutrition": "Nutrition (NUT) includes interventions improving nutritional outcomes such as treatment of malnutrition, nutrient delivery systems, dietary diversity, population nutrition support, and monitoring of nutrition outcomes, excluding specialized nutritious food formulation.",
    "Partnerships": "Partnerships (PRT) includes institutional collaborations with governments, UN agencies, private sector, academia, or donors enabling humanitarian or food security outcomes, excluding celebrity advocacy unless formally designated.",
    "Private_sector": "Private Sector (PS) includes collaborations, financing, supply chain partnerships, innovation support, or operational engagement with private companies to improve food or humanitarian outcomes, excluding general funding unless PS funded.",
    "Procurement": "Procurement (PROC) refers to sourcing, purchasing, contracting, tendering, and acquisition of goods or services, excluding supply chain unless procurement flow directly mapped.",
    "Programme_design": "Programme Design (PD) includes planning, theory of change, frameworks, targeting design, conditionality design, modality choice, and structured programme planning, excluding monitoring unless tied to PD indicators.",
    "Resilience_building": "Resilience Building (RB) refers to strengthening community or national ability to absorb, adapt, and recover from shocks using systemic or community interventions, excluding post-crisis structured resilience programming.",
    "Resilience_programming": "Resilience Programming (RP) includes structured resilience programmes, implementation design, targeting, conditionality, and execution of resilience interventions, excluding general Resilience Building unless part of RP implementation.",
    "Rome_based_agencies": "Rome-Based Agencies (RBA) refers to coordination, partnership, or policy alignment involving WFP, FAO, and IFAD, excluding general UN agency tagging unless RBA context specific.",
    "School_meals": "School Meals (SM) includes programmes delivering food or nutrition support to children in educational settings, including national school feeding initiatives, excluding general Food Assistance unless broader than SM.",
    "Seasonal_and_agricultural_monitoring": "Seasonal and Agricultural Monitoring (SAM) includes crop tracking, seasonal food production, NDVI, remote sensing, climate-linked agriculture monitoring systems excluding non-seasonal climate tools.",
    "Smallholder_agricultural_market_support": "Smallholder Agricultural Market Support (SAMS) includes interventions supporting small farmers, market linkage, pricing enablement, supply support, market access, excluding general local market monitoring unless smallholder linked.",
    "South_South_cooperation": "South-South Cooperation (SSC) includes knowledge, resource, capacity, and system exchange between Global South countries excluding COE unless COE-led.",
    "Specialized_nutritious_food": "Specialized Nutritious Food (SNF) includes formulations such as RUSF, RUTF, LNS, lipid-based supplements, therapeutic food, excluding general food fortification.",
    "Supply_chain": "Supply Chain (SC) includes end-to-end flow of goods, logistics, warehousing, demand planning, procurement flow, last-mile delivery, excluding market analysis unless tied to SC risk.",
    "Sustainable_development_goals": "Sustainable Development Goals (SDG) includes SDG alignment, SDG 2 Zero Hunger, SDG programme mapping, excluding general Food Systems unless SDG mapped.",
    "Sustainable_livelihoods_and_ecosystems": "Sustainable Livelihoods and Ecosystems (SLE) includes ecosystem protection, regenerative livelihoods, food ecosystems sustainability excluding ESS unless compliance driven.",
    "The_R4_Rural_Resilience_Initiative": "R4 Rural Resilience Initiative (R4) includes climate risk financing, insurance, resilience fund, asset creation linked to R4 excluding general resilience unless R4 tied.",
    "UN_Humanitarian_Air_Service": "UN Humanitarian Air Service (UNHAS) includes air logistics, emergency air delivery, aviation support, excluding non-humanitarian aviation.",
    "UN_agencies_and_international_institutions": "UN Agencies and International Institutions (UNII) includes UN system entities (UNHCR, UNICEF, WHO etc.) enabling humanitarian or food security context excluding RBA unless Rome specific.",
    "Zero_Hunger": "Zero Hunger (ZH) refers to strategic, programme, policy, and system interventions advancing SDG 2 Zero Hunger, excluding generic SDG alignment not tied to hunger outcomes.",
    "Social_Protection_and_Safety_Nets": "Social Protection and Safety Nets (SPSN) cover government- or partner-led systems that reduce vulnerability and protect households from shocks through predictable assistance. It includes social assistance, cash or in-kind safety nets, public works, shock-responsive social protection, registries, targeting systems, and linkages between humanitarian support and national protection systems. It excludes standalone emergency relief unless explicitly designed to connect to or strengthen national safety net systems."

}


In [ ]:
taxonomy['Academia_and_think_tanks']

In [ ]:
with open(path+'/data/taxonomy.json', 'w') as json_file:
    json.dump(taxonomy, json_file, indent=4)

In [ ]:
with open(path+'/data/taxonomy.json', 'r') as file:
    # Use json.load() to convert the file content to a dictionary
    taxonomy = json.load(file)

### Download Sentence Transformers from Hugging Face

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to(device)

In [ ]:
# Embed all definitions
topic_names = list(taxonomy.keys())
topic_definitions = list(taxonomy.values())

topic_embeddings = model.encode(topic_definitions, show_progress_bar=True)

# Store them in a dataframe for later use
taxonomy_df = pd.DataFrame({
    "topic": topic_names,
    "definition": topic_definitions,
    "embedding": list(topic_embeddings)
})

taxonomy_df.to_pickle(path+'/data/taxonomy_embeddings.pkl')

print("Taxonomy embeddings created for", len(topic_names), "topics")

### Step 2 : Create semantic representation of each report by embedding all its sentences and scoring them against all topic definition vectors. Retrieve the Top-10 topics.

In [ ]:
report_embeddings = pd.read_pickle(path+'/data/metadata_with_embeddings.pkl')

In [ ]:
# Load embedding model (384-dim, multilingual, efficient)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Pre-embed topic names (reference vectors)
topics = list(taxonomy.keys())
topic_vectors = model.encode(topics, convert_to_numpy=True)
topic_vectors = topic_vectors / np.linalg.norm(topic_vectors, axis=1, keepdims=True)  # normalize

results = []

for report_id, sentences in report_embeddings[['report_id', 'clean_sentence']].itertuples(index=False, name=None):
    
    # Embed all sentences for this report
    R = model.encode(sentences, convert_to_numpy=True)
    R = R / np.linalg.norm(R, axis=1, keepdims=True)  # normalize report vectors
    
    # Compute similarity of every sentence vs every topic
    sim_matrix = cosine_similarity(R, topic_vectors)
    
    # Aggregate per topic using max similarity (strongest evidence)
    best_sims = sim_matrix.max(axis=0)
    
    # Rank Top-10 topics
    ranked_idx = np.argsort(best_sims)[::-1][:10]
    
    top_topics = [(topics[i], best_sims[i]) for i in ranked_idx]

    # Store results
    for rank, (topic, score) in enumerate(top_topics):
        results.append({
            "report_id": report_id,
            "topic_rank": rank + 1,
            "topic": topic,
            "similarity": round(float(score), 4)
        })

# Export
out_df = pd.DataFrame(results)
out_df.to_csv(path+"/data/wfp_taxonomy_tags.csv", index=False)


print("Tagging complete. Saved to: wfp_taxonomy_tags.csv")


### Step 3: Pick the sentences corresponding to each of the top 10 topics from the cleaned text and send it to the LLM to identify the precise topics

In [ ]:
def pick_evidence_sentences(sentences, sentence_embeddings, topic_vectors, topic_names,
                           k=2, min_sent=8, max_sent=15):
    """
    Pick 8–15 sentences per report that have strongest evidence for the candidate topics.
    """
    # Stack and normalize sentence embeddings
    S = np.vstack(sentence_embeddings)
    S = S / np.linalg.norm(S, axis=1, keepdims=True)

    # Compute similarity between sentences × all topic vectors
    sim_matrix = cosine_similarity(S, topic_vectors)

    # Collect top-k sentence indices per candidate topic
    evidence_idxs = []
    for topic in topic_names:
        topic_idx = topic_names.index(topic)
        top_sent_idxs = np.argsort(sim_matrix[:, topic_idx])[::-1][:k]
        evidence_idxs.extend(top_sent_idxs)

    # Deduplicate while preserving order
    evidence_idxs = list(dict.fromkeys(evidence_idxs))

    # If not enough sentences, pad with globally strong evidence sentences
    if len(evidence_idxs) < min_sent:
        global_rank = np.argsort(sim_matrix.max(axis=1))[::-1]
        for idx in global_rank:
            if idx not in evidence_idxs:
                evidence_idxs.append(idx)
            if len(evidence_idxs) >= max_sent:
                break

    # Trim to max limit
    evidence_idxs = evidence_idxs[:max_sent]

    # Return the actual sentence strings
    return evidence_idxs


In [ ]:
import ast
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Load model again if not already loaded
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


# Load zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")


# Pre-embedded topics already exist, so we skip that part here
topics = list(taxonomy.keys())

def retrieve_top10_from_scores(best_sims):
    ranked_idx = np.argsort(best_sims)[::-1][:10]
    return [(topics[i], best_sims[i]) for i in ranked_idx]

def build_llm_prompt(evidence_sentences, candidate_topics):
    evidence_str = "\n".join(f"- {s}" for s in evidence_sentences)
    candidates_str = "\n".join(
        f"{i+1}. {topic} (similarity: {score})" for i, (topic, score) in enumerate(candidate_topics)
    )

    prompt = f"""
You are a precise humanitarian taxonomy classifier.

Report evidence sentences (most topic-bearing lines from the document):
{evidence_str}

Top-10 candidate taxonomy topics retrieved using cosine similarity:
{candidates_str}

Task:
Select the most accurate 1 to N taxonomy tags that best describe this report.

Rules:
- Only choose from the 10 candidate topic keys listed above.
- Return ONLY valid taxonomy keys from that list.
- Do NOT include explanations, commentary, or any extra text.
- Output must be a Python list of strings.

Output:
"""
    return prompt.strip()
    


In [ ]:
import ast

final_tags = []

for report_id, sentences in report_embeddings[['report_id','clean_sentence']].itertuples(index=False, name=None):

    # convert to list if needed
    if isinstance(sentences, str):
        sentences = ast.literal_eval(sentences)

    # get the 10 retrieved topics for this report from your CSV
    cand = out_df[out_df['report_id'] == report_id].sort_values('topic_rank').head(10)
    topic_names = cand['topic'].tolist()

    # flatten sentences into a small evidence string
    evidence_text = " ".join(s for s in sentences if isinstance(s, str))

    # classify against only 10 topic labels
    pred = classifier(evidence_text, topic_names, multi_label=True)

    # choose tags with highest scores (top 3–5 or threshold based)
    best = np.argsort(pred['scores'])[::-1][:5]
    tags = [pred['labels'][i] for i in best]

    final_tags.append({
        "report_id": report_id,
        "final_tags": tags
    })

# export
out_df = pd.DataFrame(final_results)
out_df.to_csv(path+"/data/final_taxonomy_tags.csv", index=False)
print("Done.")
